# 🚀 DermaSense AI - Huấn Luyện Trên Google Colab (GitHub Edition)
**Quy trình chuyên nghiệp:** GitHub → Colab → Google Drive

**⚠️ BẮT BUỘC: Bật GPU trước khi chạy!**
- `Runtime` → `Change runtime type` → `T4 GPU` → `Save`

## Bước 1: Clone mã nguồn từ GitHub
Tự động kéo code mới nhất, không cần zip hay upload thủ công!

In [ ]:
import os

REPO_URL = 'https://github.com/PHANQUOCTHANG/DermaSense_AI.git'
PROJECT_DIR = '/content/DermaSense_AI'

if os.path.exists(PROJECT_DIR):
    # Nếu đã clone rồi (ví dụ chạy lại), chỉ cần pull code mới
    %cd {PROJECT_DIR}
    !git pull origin main
    print('Da cap nhat code moi nhat tu GitHub!')
else:
    !git clone {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}
    print('Da clone thanh cong tu GitHub!')

!ls

## Bước 2: Tải dữ liệu DermNet từ Kaggle
Nếu Kaggle không cho tải file `kaggle.json`, bạn có thể nhập trực tiếp **Username** và **Key** ở ô dưới đây.

In [ ]:
import os
import json
import getpass

print("========== KẾT NỐI KAGGLE ==========")
print("Để tải ảnh về, Colab cần kết nối với tài khoản Kaggle của bạn.")
print("Bạn có muốn tải file kaggle.json lên không? (Nhập 'y' để tải file, nhập 'n' để tự gõ Username/Key)")
choice = input("Lựa chọn của bạn (y/n): ").strip().lower()

os.makedirs('/root/.kaggle', exist_ok=True)

if choice == 'y':
    from google.colab import files
    print('\nHay tai file kaggle.json cua ban len...')
    uploaded_kaggle = files.upload()
    !cp kaggle.json ~/.kaggle/
else:
    print("\n--- Nhập thủ công ---")
    print("(Bạn vào Settings trên Kaggle -> API -> Generate New Token. Copy 2 thông tin nó hiện ra)")
    username = input("Nhập Kaggle Username: ").strip()
    key = getpass.getpass("Nhập Kaggle Key (khi dán vào sẽ không hiện chữ để bảo mật, cứ Enter): ").strip()
    
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump({"username": username, "key": key}, f)

!chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle

print('\nDang tai du lieu DermNet (1.7GB). Vui lòng đợi...')
!kaggle datasets download -d shubhamgoel27/dermnet -p data/raw/dermnet_raw --unzip
print('\nTai xong!')

## Bước 3: Chuẩn bị dữ liệu (Ingestion)

In [ ]:
import shutil
import random
import pandas as pd
from pathlib import Path

dermnet_dir = Path('data/raw/dermnet_raw')
all_images = []

for split_dir in ['train', 'test']:
    split_path = dermnet_dir / split_dir
    if split_path.exists():
        for cls_dir in split_path.iterdir():
            if cls_dir.is_dir():
                for img_path in cls_dir.glob('*.*'):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                        all_images.append((img_path, cls_dir.name))

print(f'Tim thay {len(all_images)} anh. Dang xu ly...')

img_out = Path('data/raw/images')
if img_out.exists():
    shutil.rmtree(img_out)
img_out.mkdir(parents=True, exist_ok=True)

anatom_sites = ['anterior torso', 'head/neck', 'lower extremity', 'upper extremity', 'posterior torso']
symptoms = ['itch', 'bleeding', 'pain', 'none']
skin_types = ['I', 'II', 'III', 'IV', 'V', 'VI']

metadata = []
for idx, (img_path, label) in enumerate(all_images):
    img_id = f'ISIC_{idx:07d}'
    new_path = img_out / f'{img_id}{img_path.suffix}'
    shutil.copy2(img_path, new_path)
    metadata.append({
        'image_id': img_id, 'diagnosis': label,
        'age': random.randint(15, 80), 'sex': random.choice(['male', 'female']),
        'anatom_site': random.choice(anatom_sites), 'duration': random.randint(5, 365),
        'symptoms': random.choice(symptoms), 'skin_type': random.choice(skin_types),
        'family_history': random.choice(['yes', 'no'])
    })
    if (idx + 1) % 2000 == 0:
        print(f'  Da xu ly {idx+1}/{len(all_images)} anh...')

df = pd.DataFrame(metadata)
df.to_csv('data/raw/metadata.csv', index=False)
print(f'\nHoan tat! {len(all_images)} anh + metadata.csv')

## Bước 4: Cài đặt thư viện

In [ ]:
!pip install -q timm albumentations opencv-python pyyaml

## Bước 5: Kết nối Google Drive (Lưu kết quả an toàn + Resume Training)

In [ ]:
from google.colab import drive
import yaml, os

drive.mount('/content/drive')

drive_ckpt = '/content/drive/MyDrive/DermaSense_AI_Checkpoints/stage_b'
os.makedirs(drive_ckpt, exist_ok=True)

with open('configs/stage3_train_multimodal.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
cfg['training']['checkpoint_dir'] = drive_ckpt
with open('configs/stage3_train_multimodal.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f)

last_ckpt = os.path.join(drive_ckpt, 'last_checkpoint.pt')
if os.path.exists(last_ckpt):
    print('Tim thay checkpoint cu tren Drive! AI se hoc tiep tu cho cu.')
else:
    print('Chua co checkpoint cu. Bat dau huan luyen tu dau.')

print(f'Checkpoint se duoc luu tai: {drive_ckpt}')

## Bước 6: 🧠 KHỞI CHẠY HUẤN LUYỆN
Bấm Play và đợi 2-3 tiếng. Đừng tắt tab!

Nếu bị ngắt giữa chừng, lần sau chỉ cần chạy lại từ **Bước 1** → AI sẽ tự động học tiếp từ chỗ cũ.

In [ ]:
!PYTHONIOENCODING="utf-8" python -m src.pipelines.run_stage3_train_multimodal

## Bước 7: Tải file AI về máy
Hoặc vào Google Drive → `DermaSense_AI_Checkpoints/stage_b` → tải `best_model.pt`.

In [ ]:
from google.colab import files
import os

best_path = os.path.join(drive_ckpt, 'best_model.pt')
if os.path.exists(best_path):
    print(f'Tim thay best_model.pt ({os.path.getsize(best_path)/1e6:.1f} MB). Dang tai ve...')
    files.download(best_path)
else:
    print('Chua co best_model.pt. Buoc 6 chua chay xong.')